In [91]:
import sys
import os
import random
import re
import json
import pickle
import random
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import nltk
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn_crfsuite import CRF  
# nltk.download('punkt')

In [95]:
class CustomNERModel:
    def __init__(self):
        self.entity_types = set()
        self.model = None
        self.vectorizer = DictVectorizer(sparse=False)
        
    def load_data(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            try:
                data = json.loads(content)
                if isinstance(data, list):
                    return data
            except json.JSONDecodeError:
                pass
            
            data = []
            json_pattern = r'\{[^{]*"full_text":[^{]*"spans":[^{]*\}'
            json_objects = re.findall(json_pattern, content, re.DOTALL)
            
            if not json_objects:
                parts = re.split(r'}\s*,\s*\{', content)
                for i, part in enumerate(parts):
                    if not part.strip():
                        continue
                    if not part.startswith('{'):
                        part = '{' + part
                    if not part.endswith('}'):
                        part = part + '}'
                    part = re.sub(r',\s*}', '}', part)
                    try:
                        obj = json.loads(part)
                        if 'full_text' in obj and 'spans' in obj:
                            data.append(obj)
                    except json.JSONDecodeError:
                        try:
                            full_text_match = re.search(r'"full_text":\s*"([^"]*)"', part)
                            masked_match = re.search(r'"masked":\s*"([^"]*)"', part)
                            spans_match = re.search(r'"spans":\s*(\[[^\]]*\])', part)
                            if full_text_match and spans_match:
                                full_text = full_text_match.group(1).replace('\\', '')
                                masked = masked_match.group(1).replace('\\', '') if masked_match else ""
                                spans = json.loads(spans_match.group(1))
                                obj = {
                                    "full_text": full_text,
                                    "masked": masked,
                                    "spans": spans
                                }
                                data.append(obj)
                        except Exception as e:
                            print(f"Could not parse part {i}: {e}")
            else:
                for json_obj in json_objects:
                    try:
                        json_obj = re.sub(r',\s*}', '}', json_obj)
                        obj = json.loads(json_obj)
                        if 'full_text' in obj and 'spans' in obj:
                            data.append(obj)
                    except json.JSONDecodeError as e:
                        print(f"Error parsing JSON object: {e}")
            
            if not data:
                print("Warning: No valid data objects found in the file.")
            else:
                print(f"Successfully loaded {len(data)} data objects.")
            return data
        
        except Exception as e:
            print(f"Error loading data: {e}")
            return []
    
    def preprocess_data(self, data):
        all_tokens = []
        all_labels = []
        for example in data:
            if not isinstance(example, dict):
                print(f"Skipping invalid example: {type(example)}")
                continue
            full_text = example.get('full_text', '')
            spans = example.get('spans', [])
            if not full_text or not isinstance(full_text, str):
                print(f"Skipping example with invalid full_text: {full_text}")
                continue
            char_labels = ['O'] * len(full_text)
            for span in spans:
                if not isinstance(span, dict):
                    continue
                entity_type = span.get('entity_type', '')
                start_pos = span.get('start_position', 0)
                end_pos = span.get('end_position', 0)
                if not entity_type:
                    continue
                self.entity_types.add(entity_type)
                if start_pos < len(char_labels):
                    char_labels[start_pos] = f'B-{entity_type}'
                for i in range(start_pos + 1, min(end_pos, len(char_labels))):
                    char_labels[i] = f'I-{entity_type}'
            tokens = list(full_text)
            labels = char_labels
            all_tokens.append(tokens)
            all_labels.append(labels)
        if not all_tokens:
            print("Warning: No tokens generated during preprocessing")
        return all_tokens, all_labels
    
    def extract_features(self, tokens, index):
        if not tokens:
            return {'token': '', 'is_empty': True}
        token = tokens[index] if index < len(tokens) else ''
        features = {
            'token': token,
            'is_upper': token.isupper() if token else False,
            'is_title': token.istitle() if token else False,
            'is_digit': token.isdigit() if token else False,
            'is_alpha': token.isalpha() if token else False,
            'is_alnum': token.isalnum() if token else False,
        }
        if token:
            features['prefix_1'] = token[0]
            features['suffix_1'] = token[-1]
            if len(token) >= 2:
                features['prefix_2'] = token[:2]
                features['suffix_2'] = token[-2:]
        for i in range(-2, 3):
            if i == 0:
                continue
            ctx_index = index + i
            if 0 <= ctx_index < len(tokens):
                ctx_token = tokens[ctx_index]
                features[f'token_{i}'] = ctx_token
                features[f'is_upper_{i}'] = ctx_token.isupper()
                features[f'is_title_{i}'] = ctx_token.istitle()
                features[f'is_digit_{i}'] = ctx_token.isdigit()
            else:
                features[f'token_{i}'] = ''
                features[f'is_upper_{i}'] = False
                features[f'is_title_{i}'] = False
                features[f'is_digit_{i}'] = False
        return features

    def prepare_training_data(self, all_tokens, all_labels):
        X, y = [], []
        for tokens, labels in zip(all_tokens, all_labels):
            for i in range(len(tokens)):
                X.append(self.extract_features(tokens, i))
                y.append(labels[i])
        if not X:
            print("Warning: No features generated for training")
        else:
            print(f"Generated {len(X)} feature vectors for training")
        return X, y

    def train(self, X, y):
        if not X:
            raise ValueError("Training data is empty.")
        print(f"\nTraining with {len(X)} examples and {len(set(y))} unique labels")
        self.model = Pipeline([
            ('vectorizer', DictVectorizer(sparse=True)),
            ('classifier', RandomForestClassifier(n_estimators=100))
        ])
        self.model.fit(X, y)

    def predict(self, text):
        if not self.model:
            raise ValueError("Model is not trained or loaded")
        tokens = list(text)
        spans = [(i, i+1) for i in range(len(text))]
        features = [self.extract_features(tokens, i) for i in range(len(tokens))]
        predicted_labels = self.model.predict(features)
        entities = []
        for token, label, (start, end) in zip(tokens, predicted_labels, spans):
            if label.startswith('B-'):
                entity_type = label[2:]
                entities.append({'entity_type': entity_type, 'start_position': start, 'end_position': end})
            elif label.startswith('I-') and entities:
                entities[-1]['end_position'] = end
        return entities


    def anonymize_text(self, text, location_list=None, company_list=None):
        entities = self.predict(text)
        filtered_entities = []
        for entity in entities:
            entity_type = entity['entity_type'].upper()
            entity_value = text[entity['start_position']:entity['end_position']]
            should_anonymize = False
            if entity_type in {'PERSON', 'EMAIL', 'PHONE'}:
                should_anonymize = True
            elif entity_type == 'LOCATION' and location_list:
                should_anonymize = any(loc.lower() in entity_value.lower() for loc in location_list)
            elif entity_type == 'COMPANY' and company_list:
                should_anonymize = any(company.lower() in entity_value.lower() for company in company_list)
            if should_anonymize:
                filtered_entities.append(entity)
        entity_counters = {etype.upper(): set() for etype in [e['entity_type'] for e in filtered_entities]}
        for entity_type in entity_counters:
            while len(entity_counters[entity_type]) < sum(e['entity_type'].upper() == entity_type for e in filtered_entities):
                entity_counters[entity_type].add(random.randint(10000, 99999))
        for entity_type in entity_counters:
            entity_counters[entity_type] = list(entity_counters[entity_type])
        filtered_entities.sort(key=lambda x: x['start_position'], reverse=True)
        anonymized_text = text
        replacement_map = {}
        used_numbers = {etype: 0 for etype in entity_counters}
        for entity in filtered_entities:
            entity_type = entity['entity_type'].upper()
            start = entity['start_position']
            end = entity['end_position']
            value = text[start:end]
            random_number = entity_counters[entity_type][used_numbers[entity_type]]
            used_numbers[entity_type] += 1
            placeholder = f"<{entity_type}_{random_number}>"
            anonymized_text = anonymized_text[:start] + placeholder + anonymized_text[end:]
            replacement_map[placeholder] = value
        return anonymized_text, replacement_map

    def evaluate(self, X_test, y_test):
        if not self.model:
            raise ValueError("Model is not trained or loaded")
        y_pred = self.model.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        return report

    def save_model(self, model_path='custom_ner_model.pkl'):
        if not self.model:
            raise ValueError("No trained model to save")
        with open(model_path, 'wb') as f:
            pickle.dump({'model': self.model, 'entity_types': self.entity_types}, f)
        print(f"Model saved to {model_path}")

    def load_model(self, model_path='custom_ner_model.pkl'):
        try:
            with open(model_path, 'rb') as f:
                data = pickle.load(f)
                self.model = data['model']
                self.entity_types = data['entity_types']
            print(f"Model loaded from {model_path}")
            return True
        except Exception as e:
            print(f"Error loading model: {e}")
            return False


In [96]:
def main():
    
    training_file = 'dataset.txt'
    test_file = 'test.txt'
    
    location_list_file = 'locations.txt'
    company_list_file = 'companies.txt'
    
    locations = []
    companies = []
    
    if os.path.exists(location_list_file):
        try:
            with open(location_list_file, 'r', encoding='utf-8') as f:
                content = f.read()
                locations = [location.strip() for location in content.split(',') if location.strip()]
            print(f"Loaded {len(locations)} locations from {location_list_file}")
        except Exception as e:
            print(f"Error loading location list: {e}")
    else:
        print(f"Location list file {location_list_file} not found. Will not anonymize locations.")
        
    if os.path.exists(company_list_file):
        try:
            with open(company_list_file, 'r', encoding='utf-8') as f:
                content = f.read()
                companies = [company.strip() for company in content.split(',') if company.strip()]
            print(f"Loaded {len(companies)} companies from {company_list_file}")
        except Exception as e:
            print(f"Error loading company list: {e}")
    else:
        print(f"Company list file {company_list_file} not found. Will not anonymize companies.")
    
    if not os.path.exists(training_file):
        print(f"Error: Training file '{training_file}' not found!")
        return
        
    if not os.path.exists(test_file):
        print(f"Warning: Test file '{test_file}' not found. Skipping test phase.")
        test_file = None
    
    print(f"Using training data from: {training_file}")
    if test_file:
        print(f"Using test data from: {test_file}")
    
    ner_model = CustomNERModel()
    
    model_path = os.path.splitext(training_file)[0] + '_model.pkl'
    if os.path.exists(model_path):
        if ner_model.load_model(model_path):
            print("Model loaded successfully!")
        else:
            print("Failed to load model. Training a new one...")
            train_new_model = True
    else:
        print("No existing model found. Training a new one...")
        train_new_model = True
    
    if 'train_new_model' in locals() and train_new_model:
        print("Loading and preprocessing training data...")
        data = ner_model.load_data(training_file)
        print(f"Loaded {len(data)} training examples.")
        all_tokens, all_labels = ner_model.preprocess_data(data)
        
        if not all_tokens:
            print("Error: No tokens extracted from training data.")
            return
            
        X, y = ner_model.prepare_training_data(all_tokens, all_labels)
        
        if not X:
            print("Error: No features extracted for training.")
            return
            
        indices = list(range(len(X)))
        if len(indices) > 1:  
            random.shuffle(indices)
        split_point = max(1, int(0.8 * len(indices))) 
        
        train_indices = indices[:split_point]
        test_indices = indices[split_point:]
        
        X_train = [X[i] for i in train_indices]
        y_train = [y[i] for i in train_indices]
        X_test = [X[i] for i in test_indices]
        y_test = [y[i] for i in test_indices]
        
        print(f"Training data split: {len(X_train)} train, {len(X_test)} validation examples")
        
        try:
            print("Training the NER model...")
            ner_model.train(X_train, y_train)
            
            if X_test and y_test:
                print("\nModel Evaluation on Validation Set:")
                evaluation = ner_model.evaluate(X_test, y_test)
                print(f"Overall accuracy: {evaluation['accuracy']:.4f}")

                for label, metrics in evaluation.items():
                    if label not in ['accuracy', 'macro avg', 'weighted avg']:
                        print(f"{label}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1-score={metrics['f1-score']:.4f}")
            
            ner_model.save_model(model_path)
            
        except Exception as e:
            print(f"Error during model training or evaluation: {e}")
    
    if test_file and os.path.exists(test_file):
        print(f"\nProcessing test data from: {test_file}")
        try:
            with open(test_file, 'r') as f:
                test_text = f.read()
            
            if not test_text.strip():
                print("Warning: Test file is empty")
            else:
                entities = ner_model.predict(test_text)
                
                if entities:
                    for entity in entities:
                        value = test_text[entity['start_position']:entity['end_position']]
                        print(f"{entity['entity_type']}: {value} ({entity['start_position']}-{entity['end_position']})")
                else:
                    print("No entities detected.")
        
        except Exception as e:
            print(f"Error processing test file: {e}")
            import traceback
            traceback.print_exc()

if __name__ == "__main__":
    main()

Loaded 209 locations from locations.txt
Loaded 435 companies from companies.txt
Using training data from: dataset.txt
Using test data from: test.txt
No existing model found. Training a new one...
Loading and preprocessing training data...
Loaded 165 training examples.
Generated 34379 feature vectors for training
Training data split: 27503 train, 6876 validation examples
Training the NER model...

Training with 27503 examples and 91 unique labels

Model Evaluation on Validation Set:


c:\DB\ner_model\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\DB\ner_model\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\DB\ner_model\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Overall accuracy: 0.7744
B-AWARD: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-CURRENCY: Precision=0.5000, Recall=0.5000, F1-score=0.5000
B-DATE: Precision=0.3333, Recall=0.2500, F1-score=0.2857
B-DEMONYM: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-DEPARTMENT: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-DOMAIN_NAME: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-DURATION: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-EMAIL: Precision=1.0000, Recall=0.2500, F1-score=0.4000
B-ENVIROMENTAL_IMPACT: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-ENVIRONMENTAL_FACTOR: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-ENVIRONMENTAL_IMPACT: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-EVENT: Precision=1.0000, Recall=0.1429, F1-score=0.2500
B-GPE: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-INDUSTRY_SECTOR: Precision=0.0000, Recall=0.0000, F1-score=0.0000
B-INGREDIENT: Precision=0.6667, Recall=0.2500, F1-score=0.3636
B-LOCATION: Precision=0.4286, 

In [76]:
from collections import Counter
print("Label distribution:", Counter(y))

Label distribution: Counter({'O': 4663, 'I-ORGANIZATION': 147, 'I-PERSON': 141, 'B-ORGANIZATION': 118, 'I-LOCATION': 96, 'B-PERSON': 70, 'I-EVENT': 66, 'B-LOCATION': 57, 'I-DATE': 55, 'I-PRODUCT': 50, 'I-TITLE': 43, 'B-TITLE': 34, 'B-PRODUCT': 28, 'I-INDUSTRY_SECTOR': 24, 'I-PHONE_NUMBER': 23, 'I-EMAIL': 19, 'I-INGREDIENT': 19, 'I-PERCENTAGE': 17, 'I-MONEY': 15, 'I-STUDY_TYPE': 12, 'I-DOMAIN_NAME': 11, 'I-INITIATIVE': 11, 'I-TECHNOLOGY': 10, 'B-DATE': 9, 'B-PERCENTAGE': 9, 'I-AWARD': 9, 'B-TECHNOLOGY': 8, 'I-DURATION': 6, 'B-MONEY': 6, 'B-EVENT': 5, 'B-INGREDIENT': 5, 'B-INDUSTRY_SECTOR': 5, 'I-RETAILER': 4, 'B-AWARD': 4, 'I-PRODUCT_TYPE': 4, 'I-WASTE_TYPE': 4, 'I-INDUSTRY_TREND': 4, 'B-PRODUCT_TYPE': 3, 'I-APPLICATION': 3, 'I-PLATFORM': 3, 'B-GPE': 2, 'B-CURRENCY': 2, 'I-DEPARTMENT': 2, 'I-FEATURE': 2, 'I-CERTIFICATION_TYPE': 2, 'B-SKIN_CONCERN': 2, 'B-NUMBER': 2, 'B-DURATION': 2, 'I-SKIN_BENEFIT': 2, 'B-PHONE_NUMBER': 1, 'B-DEPARTMENT': 1, 'B-REGULATORY_BODY': 1, 'B-VOLUME': 1, 'B-PR

In [1]:
import json
import re
import random
import pickle
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [29]:
class CustomNERModel:
    def __init__(self):
        self.target_entity_types = {'PERSON', 'ORGANIZATION', 'LOCATION', 'EMAIL', 
                                    'PHONE_NUMBER', 'COUNTRY', 'TITLE', 'COMPANY'}
 
        self.entity_mappings = {'COMPANY': 'ORGANIZATION'}  
        self.entity_types = set()
        self.model = None
        self.vectorizer = DictVectorizer(sparse=False)
        
    def load_data(self, file_path, is_json=True):
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            if not is_json:
                paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]
                if not paragraphs:
                    paragraphs = [p.strip() for p in content.split('\n') if p.strip()]
                    if not paragraphs:
                        paragraphs = [content]
                
                data = []
                for paragraph in paragraphs:
                    data.append({
                        "full_text": paragraph,
                        "spans": []
                    })
                print(f"Loaded {len(data)} text paragraphs from {file_path}")
                return data
            
            try:
                data = json.loads(content)
                if isinstance(data, list):
                    return data
            except json.JSONDecodeError:
                pass
            
            data = []
            json_pattern = r'\{[^{]*"full_text":[^{]*"spans":[^{]*\}'
            json_objects = re.findall(json_pattern, content, re.DOTALL)
            
            if not json_objects:
                parts = re.split(r'}\s*,\s*\{', content)
                for i, part in enumerate(parts):
                    if not part.strip():
                        continue
                    if not part.startswith('{'):
                        part = '{' + part
                    if not part.endswith('}'):
                        part = part + '}'
                    part = re.sub(r',\s*}', '}', part)
                    try:
                        obj = json.loads(part)
                        if 'full_text' in obj and 'spans' in obj:
                            data.append(obj)
                    except json.JSONDecodeError:
                        try:
                            full_text_match = re.search(r'"full_text":\s*"([^"]*)"', part)
                            masked_match = re.search(r'"masked":\s*"([^"]*)"', part)
                            spans_match = re.search(r'"spans":\s*(\[[^\]]*\])', part)
                            if full_text_match and spans_match:
                                full_text = full_text_match.group(1).replace('\\', '')
                                masked = masked_match.group(1).replace('\\', '') if masked_match else ""
                                spans = json.loads(spans_match.group(1))
                                obj = {
                                    "full_text": full_text,
                                    "masked": masked,
                                    "spans": spans
                                }
                                data.append(obj)
                        except Exception as e:
                            print(f"Could not parse part {i}: {e}")
            else:
                for json_obj in json_objects:
                    try:
                        json_obj = re.sub(r',\s*}', '}', json_obj)
                        obj = json.loads(json_obj)
                        if 'full_text' in obj and 'spans' in obj:
                            data.append(obj)
                    except json.JSONDecodeError as e:
                        print(f"Error parsing JSON object: {e}")
            
            if not data:
                print("Warning: No valid data objects found in the file.")
            else:
                print(f"Successfully loaded {len(data)} data objects.")
            return data
        
        except Exception as e:
            print(f"Error loading data: {e}")
            return []

    def load_entity_list(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                
            if ',' in content:
                entities = [entity.strip() for entity in content.split(',') if entity.strip()]
            else:
                entities = [entity.strip() for entity in content.split('\n') if entity.strip()]
                
            print(f"Loaded {len(entities)} entities from {file_path}")
            return entities
        except Exception as e:
            print(f"Error loading entity list from {file_path}: {e}")
            return []
    
    def preprocess_data(self, data):
        all_tokens = []
        all_labels = []
        
        entity_counts = {etype: 0 for etype in self.target_entity_types}
        non_entity_count = 0  
        
        for example in data:
            if not isinstance(example, dict):
                print(f"Skipping invalid example: {type(example)}")
                continue
                
            full_text = example.get('full_text', '')
            spans = example.get('spans', [])
            
            if not full_text or not isinstance(full_text, str):
                print(f"Skipping example with invalid full_text: {full_text}")
                continue
                
            char_labels = ['O'] * len(full_text)
            
            for span in spans:
                if not isinstance(span, dict):
                    continue
                    
                entity_type = span.get('entity_type', '')
                start_pos = span.get('start_position', 0)
                end_pos = span.get('end_position', 0)
                
                entity_type = entity_type.upper()
                if entity_type.startswith('B-') or entity_type.startswith('I-'):
                    entity_type = entity_type[2:]
                
                if entity_type in self.entity_mappings:
                    entity_type = self.entity_mappings[entity_type]
                
                if not entity_type or entity_type not in self.target_entity_types:
                    continue
                    
                self.entity_types.add(entity_type)
                entity_counts[entity_type] += 1
                
                for i in range(start_pos, min(end_pos, len(char_labels))):
                    char_labels[i] = entity_type
                    
            non_entity_count += char_labels.count('O')
                    
            tokens = list(full_text)
            labels = char_labels
            all_tokens.append(tokens)
            all_labels.append(labels)
            
        if not all_tokens:
            print("Warning: No tokens generated during preprocessing")
        else:
            print("Entity type counts in dataset:")
            for etype, count in sorted(entity_counts.items()):
                if count > 0:
                    print(f"  {etype}: {count}")
            print(f"  Non-entity tokens: {non_entity_count}")
                    
        return all_tokens, all_labels
    
    def extract_features(self, tokens, index):
        if not tokens:
            return {'token': '', 'is_empty': True}
            
        token = tokens[index] if index < len(tokens) else ''
        features = {
            'token': token,
            'is_upper': token.isupper() if token else False,
            'is_title': token.istitle() if token else False,
            'is_digit': token.isdigit() if token else False,
            'is_alpha': token.isalpha() if token else False,
            'is_alnum': token.isalnum() if token else False,
            'is_punct': not token.isalnum() if token else False,
            'length': len(token) if token else 0,
        }
        
        if token:
            features['prefix_1'] = token[0]
            features['suffix_1'] = token[-1]
            if len(token) >= 2:
                features['prefix_2'] = token[:2]
                features['suffix_2'] = token[-2:]
            if len(token) >= 3:
                features['prefix_3'] = token[:3]
                features['suffix_3'] = token[-3:]
                
        # Context window features
        for i in range(-3, 4): 
            if i == 0:
                continue
            ctx_index = index + i
            if 0 <= ctx_index < len(tokens):
                ctx_token = tokens[ctx_index]
                features[f'token_{i}'] = ctx_token
                features[f'is_upper_{i}'] = ctx_token.isupper()
                features[f'is_title_{i}'] = ctx_token.istitle()
                features[f'is_digit_{i}'] = ctx_token.isdigit()
                features[f'is_punct_{i}'] = not ctx_token.isalnum()
            else:
                features[f'token_{i}'] = ''
                features[f'is_upper_{i}'] = False
                features[f'is_title_{i}'] = False
                features[f'is_digit_{i}'] = False
                features[f'is_punct_{i}'] = False
                
        # Added bigram and trigram features
        if index > 0 and index < len(tokens) - 1:
            features['bigram_prev'] = tokens[index-1] + tokens[index]
            features['bigram_next'] = tokens[index] + tokens[index+1]
        if index > 1 and index < len(tokens) - 2:
            features['trigram'] = tokens[index-1] + tokens[index] + tokens[index+1]
            
        return features

    def prepare_training_data(self, all_tokens, all_labels):
        X, y = [], []
        for tokens, labels in zip(all_tokens, all_labels):
            for i in range(len(tokens)):
                X.append(self.extract_features(tokens, i))
                y.append(labels[i])
                
        if not X:
            print("Warning: No features generated for training")
        else:
            print(f"Generated {len(X)} feature vectors for training")
            
        return X, y

    def train(self, X, y, tune_hyperparams=False):
        if not X:
            raise ValueError("Training data is empty.")
            
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
        print(f"Training data split: {len(X_train)} train, {len(X_val)} validation examples")
        
        print(f"\nTraining with {len(X_train)} examples and {len(set(y_train))} unique labels")
        
        if tune_hyperparams:
            pipeline = Pipeline([
                ('vectorizer', DictVectorizer(sparse=True)),
                ('classifier', RandomForestClassifier(random_state=42))
            ])
            
            param_grid = {
                'classifier__n_estimators': [100, 200],
                'classifier__max_depth': [15, 20, None],
                'classifier__min_samples_split': [2, 5],
                'classifier__min_samples_leaf': [1, 2]
            }
            
            print("Performing hyperparameter tuning")
            grid_search = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=-1, scoring='f1_weighted')
            grid_search.fit(X_train, y_train)
            
            print(f"Best parameters: {grid_search.best_params_}")
            self.model = grid_search.best_estimator_
        else:
            self.model = Pipeline([
                ('vectorizer', DictVectorizer(sparse=True)),
                ('classifier', RandomForestClassifier(n_estimators=200, max_depth=20,
                                                     min_samples_split=2, min_samples_leaf=1,
                                                     random_state=42, n_jobs=-1))
            ])
            self.model.fit(X_train, y_train)
        
        val_predictions = self.model.predict(X_val)
        self.report_focused_metrics(y_val, val_predictions)
        
        self.model.fit(X, y)

    def report_focused_metrics(self, y_true, y_pred):
        report = classification_report(y_true, y_pred, output_dict=True)
        
        print("Model Evaluation on Validation Set:")
        print(f"Overall accuracy: {report['accuracy']:.4f}")
        
        for label, metrics in sorted(report.items()):
            if label == 'accuracy' or label == 'macro avg' or label == 'weighted avg' or label == 'samples avg':
                continue
            if label == 'O':
                continue  
            if label in self.target_entity_types:
                precision = metrics['precision']
                recall = metrics['recall']
                f1 = metrics['f1-score']
                print(f"{label}: Precision={precision:.4f}, Recall={recall:.4f}, F1-score={f1:.4f}")

    def predict(self, text):
        if not self.model:
            raise ValueError("Model is not trained or loaded")
            
        tokens = list(text)
        features = [self.extract_features(tokens, i) for i in range(len(tokens))]
        predicted_labels = self.model.predict(features)
        
        entities = []
        current_entity = None
        
        for i, (token, label) in enumerate(zip(tokens, predicted_labels)):
            if label in self.entity_mappings:
                label = self.entity_mappings[label]
                
            if label != 'O' and label in self.target_entity_types:
                if current_entity is None or current_entity['entity_type'] != label:
                    current_entity = {'entity_type': label, 'start_position': i, 'end_position': i+1}
                    entities.append(current_entity)
                else:
                    current_entity['end_position'] = i+1
            else:
                current_entity = None
        
        return entities

    def anonymize_text(self, text, location_list=None, company_list=None):
        
        entities = self.predict(text)
        
        filtered_entities = entities
        if location_list or company_list:
            for list_type, entity_list in [("LOCATION", location_list), ("ORGANIZATION", company_list)]:
                if not entity_list:
                    continue
                    
                for item in entity_list:
                    item = item.strip()
                    if not item:
                        continue
                        
                    pattern = r'\b' + re.escape(item) + r'\b'
                    for match in re.finditer(pattern, text, re.IGNORECASE):
                        start = match.start()
                        end = match.end()
                        
                        is_contained = False
                        for entity in filtered_entities:
                            if (start >= entity['start_position'] and 
                                end <= entity['end_position']):
                                is_contained = True
                                break
                                
                        if not is_contained:
                            filtered_entities.append({
                                'entity_type': list_type,
                                'start_position': start,
                                'end_position': end
                            })
        
        filtered_entities.sort(key=lambda x: x['start_position'], reverse=True)
        
        entity_counters = {etype.upper(): 1 for etype in self.target_entity_types}
        anonymized_text = text
        replacement_map = {}
        
        for entity in filtered_entities:
            entity_type = entity['entity_type'].upper()
            start = entity['start_position']
            end = entity['end_position']
            original_text = text[start:end]
            
            placeholder = f"<{entity_type}_{entity_counters[entity_type]}>"
            entity_counters[entity_type] += 1
            
            anonymized_text = anonymized_text[:start] + placeholder + anonymized_text[end:]
            replacement_map[placeholder] = original_text
            
        return anonymized_text, replacement_map

    def evaluate(self, X_test, y_test):
        if not self.model:
            raise ValueError("Model is not trained or loaded")
            
        y_pred = self.model.predict(X_test)
        self.report_focused_metrics(y_test, y_pred)
        
        return classification_report(y_test, y_pred, output_dict=True)

    def save_model(self, model_path='custom_ner_model.pkl'):
        if not self.model:
            raise ValueError("No trained model to save")
            
        with open(model_path, 'wb') as f:
            pickle.dump({
                'model': self.model, 
                'entity_types': self.entity_types,
                'target_entity_types': self.target_entity_types
            }, f)
            
        print(f"Model saved to {model_path}")

    def load_model(self, model_path='custom_ner_model.pkl'):
        try:
            with open(model_path, 'rb') as f:
                data = pickle.load(f)
                self.model = data['model']
                self.entity_types = data['entity_types']
                if 'target_entity_types' in data:
                    self.target_entity_types = data['target_entity_types']
                    
            print(f"Model loaded from {model_path}")
            return True
        except Exception as e:
            print(f"Error loading model: {e}")
            return False

In [44]:
import os
import sys

def main():
    ner_model = CustomNERModel()
    
    try:
        locations = ner_model.load_entity_list("locations.txt")
    except Exception as e:
        locations = []
        
    try:
        companies = ner_model.load_entity_list("companies.txt")
    except Exception as e:
        companies = []
    
    train_file = "dataset_new1.txt"
    test_file = "test.txt"
    model_file = "custom_ner_model.pkl"
    
    print(f"Using training data from: {train_file}")
    print(f"Using test data from: {test_file}")
    
    if os.path.exists(model_file):
        ner_model.load_model(model_file)
    else:
        print("No existing model found. Training a new one.")
        
        train_data = ner_model.load_data(train_file)
        print(f"Loaded {len(train_data)} training examples.")
        
        tokens, labels = ner_model.preprocess_data(train_data)
        X, y = ner_model.prepare_training_data(tokens, labels)
        
        ner_model.train(X, y, tune_hyperparams=False)
        
        ner_model.save_model(model_file)
    
    test_data = ner_model.load_data(test_file, is_json=False)
    
    if test_data:
        sample_text = test_data[0].get('full_text', '')
        if sample_text:
            # Anonymize text
            anonymized_text, replacements = ner_model.anonymize_text(sample_text, location_list=locations, company_list=companies)
            
            print("\nSample text:")
            print(sample_text)
            print("\nAnonymized text:")
            print(anonymized_text)
            print("\nReplacements:")
            for placeholder, original in replacements.items():
                print(f"{placeholder}: {original}")
    else:
        print("No test data available.")

if __name__ == "__main__":
    main()

Loaded 210 entities from locations.txt
Loaded 435 entities from companies.txt
Using training data from: dataset_new1.txt
Using test data from: test.txt
Model loaded from custom_ner_model.pkl
Loaded 1 text paragraphs from test.txt

Sample text:
16 February 2024 
CEO Mr. Roberts
Equities 
+7927156839
Email: <saurabh.mishra2(at)gds(dot)com>
Apple
Netflix
London, UK

Anonymized text:
16 February 2024 
CEO Mr. Roberts
Equities 
+<PHONE_NUMBER_1>9
Email: <saurabh.mishra2(at)gds(dot)com>
<ORGANIZATION_2>
<ORGANIZATION_1>
London, <LOCATION_1>

Replacements:
<LOCATION_1>: UK
<ORGANIZATION_1>: Netflix
<ORGANIZATION_2>: Apple
<PHONE_NUMBER_1>: 792715683


In [ ]:
class Improved_CustomNERModel:
    def __init__(self):
        self.target_entity_types = {'PERSON', 'ORGANIZATION', 'LOCATION', 'EMAIL', 
                                    'PHONE_NUMBER', 'COUNTRY', 'COMPANY'}
     
        self.entity_mappings = {'COMPANY': 'ORGANIZATION'}  
        self.entity_types = set()
        self.model = None
        self.vectorizer = DictVectorizer(sparse=False)
        
        self.min_entity_length = {
            'PERSON': 3,
            'ORGANIZATION': 3,
            'LOCATION': 3,
            'EMAIL': 5,
            'PHONE_NUMBER': 6,
            'COUNTRY': 3,
            'COMPANY': 3
        }
        
        self.confidence_threshold = 0.65
        
        self.gazetteers = {
            'PERSON': set(),
            'ORGANIZATION': set(),
            'LOCATION': set(),
            'COUNTRY': set()
        }
        
        self.patterns = {
            'EMAIL': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            'PHONE_NUMBER': r'\b(?:\+\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b',
        }
        
    def load_data(self, file_path, is_json=True):
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            if not is_json:
                paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]
                if not paragraphs:
                    paragraphs = [p.strip() for p in content.split('\n') if p.strip()]
                    if not paragraphs:
                        paragraphs = [content]
                
                data = []
                for paragraph in paragraphs:
                    data.append({
                        "full_text": paragraph,
                        "spans": []
                    })
                print(f"Loaded {len(data)} text paragraphs from {file_path}")
                return data
            
            try:
                data = json.loads(content)
                if isinstance(data, list):
                    return data
            except json.JSONDecodeError:
                pass
            
            data = []
            json_pattern = r'\{[^{]*"full_text":[^{]*"spans":[^{]*\}'
            json_objects = re.findall(json_pattern, content, re.DOTALL)
            
            if not json_objects:
                parts = re.split(r'}\s*,\s*\{', content)
                for i, part in enumerate(parts):
                    if not part.strip():
                        continue
                    if not part.startswith('{'):
                        part = '{' + part
                    if not part.endswith('}'):
                        part = part + '}'
                    part = re.sub(r',\s*}', '}', part)
                    try:
                        obj = json.loads(part)
                        if 'full_text' in obj and 'spans' in obj:
                            data.append(obj)
                    except json.JSONDecodeError:
                        try:
                            full_text_match = re.search(r'"full_text":\s*"([^"]*)"', part)
                            masked_match = re.search(r'"masked":\s*"([^"]*)"', part)
                            spans_match = re.search(r'"spans":\s*(\[[^\]]*\])', part)
                            if full_text_match and spans_match:
                                full_text = full_text_match.group(1).replace('\\', '')
                                masked = masked_match.group(1).replace('\\', '') if masked_match else ""
                                spans = json.loads(spans_match.group(1))
                                obj = {
                                    "full_text": full_text,
                                    "masked": masked,
                                    "spans": spans
                                }
                                data.append(obj)
                        except Exception as e:
                            print(f"Could not parse part {i}: {e}")
            else:
                for json_obj in json_objects:
                    try:
                        json_obj = re.sub(r',\s*}', '}', json_obj)
                        obj = json.loads(json_obj)
                        if 'full_text' in obj and 'spans' in obj:
                            data.append(obj)
                    except json.JSONDecodeError as e:
                        print(f"Error parsing JSON object: {e}")
            
            if not data:
                print("Warning: No valid data objects found in the file.")
            return data
        
        except Exception as e:
            print(f"Error loading data: {e}")

    def load_entity_list(self, file_path, entity_type=None):
      
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                
            if ',' in content:
                entities = [entity.strip() for entity in content.split(',') if entity.strip()]
            else:
                entities = [entity.strip() for entity in content.split('\n') if entity.strip()]
            
            min_length = 2  
            if entity_type and entity_type.upper() in self.min_entity_length:
                min_length = self.min_entity_length[entity_type.upper()]
                
            entities = [entity for entity in entities if len(entity) >= min_length]
            
            if entity_type and entity_type.upper() in self.gazetteers:
                normalized_type = entity_type.upper()
                if normalized_type in self.entity_mappings:
                    normalized_type = self.entity_mappings[normalized_type]
                self.gazetteers[normalized_type].update(entities)
                print(f"Added {len(entities)} entries to {normalized_type} gazetteer")
                
            print(f"Loaded {len(entities)} entities from {file_path}")
            return entities
        except Exception as e:
            print(f"Error loading entity list from {file_path}: {e}")
            return []
    
    def preprocess_data(self, data):
        all_tokens = []
        all_labels = []
        
        entity_counts = {etype: 0 for etype in self.target_entity_types}
        non_entity_count = 0 
        
        for example in data:
            if not isinstance(example, dict):
                continue
                
            full_text = example.get('full_text', '')
            spans = example.get('spans', [])
            
            for span in spans:
                if not isinstance(span, dict):
                    continue
                    
                entity_type = span.get('entity_type', '')
                start_pos = span.get('start_position', 0)
                end_pos = span.get('end_position', 0)
                
                entity_type = entity_type.upper()
                if entity_type.startswith('B-') or entity_type.startswith('I-'):
                    entity_type = entity_type[2:]
                
                if entity_type in self.entity_mappings:
                    entity_type = self.entity_mappings[entity_type]
                
                if not entity_type or entity_type not in self.target_entity_types:
                    continue

                if entity_type in self.gazetteers and 0 <= start_pos < end_pos <= len(full_text):
                    entity_text = full_text[start_pos:end_pos].strip()
                    min_length = self.min_entity_length.get(entity_type, 2)
                    if entity_text and len(entity_text) >= min_length:
                        self.gazetteers[entity_type].add(entity_text)
        
        for example in data:
            if not isinstance(example, dict):
                print(f"Skipping invalid example: {type(example)}")
                continue
                
            full_text = example.get('full_text', '')
            spans = example.get('spans', [])
            
            if not full_text or not isinstance(full_text, str):
                print(f"Skipping example with invalid full_text: {full_text}")
                continue
                
            char_labels = ['O'] * len(full_text)
            
            for span in spans:
                if not isinstance(span, dict):
                    continue
                    
                entity_type = span.get('entity_type', '')
                start_pos = span.get('start_position', 0)
                end_pos = span.get('end_position', 0)
                
                entity_type = entity_type.upper()
                if entity_type.startswith('B-') or entity_type.startswith('I-'):
                    entity_type = entity_type[2:]
                
                if entity_type in self.entity_mappings:
                    entity_type = self.entity_mappings[entity_type]
                
                if not entity_type or entity_type not in self.target_entity_types:
                    continue
                
                entity_text = full_text[start_pos:end_pos].strip()
                min_length = self.min_entity_length.get(entity_type, 2)
                if len(entity_text) < min_length:
                    continue
                    
                self.entity_types.add(entity_type)
                entity_counts[entity_type] += 1
                
                for i in range(start_pos, min(end_pos, len(char_labels))):
                    char_labels[i] = entity_type
                    
            non_entity_count += char_labels.count('O')
                    
            tokens = list(full_text)
            labels = char_labels
            all_tokens.append(tokens)
            all_labels.append(labels)
            
        if not all_tokens:
            print("Warning: No tokens generated during preprocessing")
        else:
            print("Entity type counts in dataset:")
            for etype, count in sorted(entity_counts.items()):
                if count > 0:
                    print(f"  {etype}: {count}")
            print(f"  Non-entity tokens: {non_entity_count}")
                    
        return all_tokens, all_labels
    
    def extract_features(self, tokens, index):
        if not tokens:
            return {'token': '', 'is_empty': True}
            
        token = tokens[index] if index < len(tokens) else ''
        features = {
            'token': token,
            'is_upper': token.isupper() if token else False,
            'is_title': token.istitle() if token else False,
            'is_digit': token.isdigit() if token else False,
            'is_alpha': token.isalpha() if token else False,
            'is_alnum': token.isalnum() if token else False,
            'is_punct': not token.isalnum() if token else False,
            'length': len(token) if token else 0,
        }
        
        if token:
            features['prefix_1'] = token[0]
            features['suffix_1'] = token[-1]
            if len(token) >= 2:
                features['prefix_2'] = token[:2]
                features['suffix_2'] = token[-2:]
            if len(token) >= 3:
                features['prefix_3'] = token[:3]
                features['suffix_3'] = token[-3:]
                
        for i in range(-3, 4): 
            if i == 0:
                continue
            ctx_index = index + i
            if 0 <= ctx_index < len(tokens):
                ctx_token = tokens[ctx_index]
                features[f'token_{i}'] = ctx_token
                features[f'is_upper_{i}'] = ctx_token.isupper()
                features[f'is_title_{i}'] = ctx_token.istitle()
                features[f'is_digit_{i}'] = ctx_token.isdigit()
                features[f'is_punct_{i}'] = not ctx_token.isalnum()
            else:
                features[f'token_{i}'] = ''
                features[f'is_upper_{i}'] = False
                features[f'is_title_{i}'] = False
                features[f'is_digit_{i}'] = False
                features[f'is_punct_{i}'] = False
                
        # Added bigram and trigram features
        if index > 0 and index < len(tokens) - 1:
            features['bigram_prev'] = tokens[index-1] + tokens[index]
            features['bigram_next'] = tokens[index] + tokens[index+1]
        if index > 1 and index < len(tokens) - 2:
            features['trigram'] = tokens[index-1] + tokens[index] + tokens[index+1]
        
        token_str = ''.join(tokens[max(0, index-5):min(len(tokens), index+5)])
        for pattern_type, pattern in self.patterns.items():
            features[f'pattern_{pattern_type}'] = bool(re.search(pattern, token_str))
        
        for entity_type, entity_set in self.gazetteers.items():
            features[f'in_gazetteer_{entity_type}'] = False
            
            context_start = max(0, index - 3)
            context_end = min(len(tokens), index + 4)
            context = ''.join(tokens[context_start:context_end])
            
            for entity in entity_set:
                if entity and entity.lower() in context.lower():
                    features[f'in_gazetteer_{entity_type}'] = True
                    break
            
        return features

    def prepare_training_data(self, all_tokens, all_labels):

        X, y = [], []
        for tokens, labels in zip(all_tokens, all_labels):
            for i in range(len(tokens)):
                X.append(self.extract_features(tokens, i))
                y.append(labels[i])
                
        if not X:
            print("Warning: No features generated for training")
        else:
            print(f"Generated {len(X)} feature vectors for training")
            
        return X, y

    def create_class_weights(self, y):
        
        counts = Counter(y)
        total = len(y)
        weights = {}
        
        for cls, count in counts.items():
            weights[cls] = total / (len(counts) * count)
            
        return weights

    def train(self, X, y):
  
        class_weights = self.create_class_weights(y)
        print("Using class weights to handle imbalanced data:")
        for cls, weight in sorted(class_weights.items()):
            print(f"  {cls}: {weight:.4f}")
            
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
        print(f"Training data split: {len(X_train)} train, {len(X_val)} validation examples")
        
        print(f"\nTraining with {len(X_train)} examples and {len(set(y_train))} unique labels")
        
        self.model = Pipeline([
            ('vectorizer', DictVectorizer(sparse=True)),
            ('classifier', RandomForestClassifier(
                n_estimators=300, 
                max_depth=None,
                min_samples_split=2, 
                min_samples_leaf=1,
                class_weight='balanced',
                random_state=42, 
                n_jobs=-1))
        ])
        self.model.fit(X_train, y_train)
        
        val_predictions = self.model.predict(X_val)
        self.report_focused_metrics(y_val, val_predictions)
        
        self.model.fit(X, y)

    def report_focused_metrics(self, y_true, y_pred):

        report = classification_report(y_true, y_pred, output_dict=True)
        
        print("Model Evaluation on Validation Set:")
        print(f"Overall accuracy: {report['accuracy']:.4f}")
        
        for label, metrics in sorted(report.items()):
            if label == 'accuracy' or label == 'macro avg' or label == 'weighted avg' or label == 'samples avg':
                continue
            if label == 'O':
                continue  
            if label in self.target_entity_types:
                precision = metrics['precision']
                recall = metrics['recall']
                f1 = metrics['f1-score']
                print(f"{label}: Precision={precision:.4f}, Recall={recall:.4f}, F1-score={f1:.4f}")

    def rule_based_extraction(self, text):
        entities = []
        
        for entity_type, pattern in self.patterns.items():
            for match in re.finditer(pattern, text):
                start = match.start()
                end = match.end()
                entity_text = text[start:end].strip()
                
                # Check minimum length
                min_length = self.min_entity_length.get(entity_type, 2)
                if len(entity_text) < min_length:
                    continue
                    
                entities.append({
                    'entity_type': entity_type,
                    'start_position': start,
                    'end_position': end,
                    'confidence': 1.0  # Rule-based matches
                })
        
        for entity_type, gazetteer in self.gazetteers.items():
            if not gazetteer:
                continue
                
            sorted_entities = sorted(gazetteer, key=len, reverse=True)
            
            for entity in sorted_entities:
                min_length = self.min_entity_length.get(entity_type, 2)
                if len(entity) < min_length: 
                    continue
                    
                pattern = r'\b' + re.escape(entity) + r'\b'
                for match in re.finditer(pattern, text, re.IGNORECASE):
                    start = match.start()
                    end = match.end()
                    
                    overlap = False
                    for existing in entities:
                        if (start < existing['end_position'] and 
                            end > existing['start_position']):
                            overlap = True
                            break
                    
                    if not overlap:
                        entities.append({
                            'entity_type': entity_type,
                            'start_position': start,
                            'end_position': end,
                            'confidence': 0.9  # Gazetteer matches 
                        })
        
        return entities

    def predict(self, text):
        
        rule_based_entities = self.rule_based_extraction(text)
        
        masked_positions = set()
        for entity in rule_based_entities:
            for i in range(entity['start_position'], entity['end_position']):
                masked_positions.add(i)
        
        tokens = list(text)
        features = []
        indices = []
        
        for i in range(len(tokens)):
            if i not in masked_positions:
                features.append(self.extract_features(tokens, i))
                indices.append(i)
        
        ml_entities = []
        if features and hasattr(self.model, 'predict_proba'):
            predicted_labels = self.model.predict(features)
            predicted_probs = self.model.predict_proba(features)
            
            current_entity = None
            entity_confidences = []
            
            for i, (index, label, probs) in enumerate(zip(indices, predicted_labels, predicted_probs)):
                if label in self.entity_mappings:
                    label = self.entity_mappings[label]
                
                label_idx = list(self.model.classes_).index(label) if label in self.model.classes_ else -1
                confidence = probs[label_idx] if label_idx >= 0 else 0
                    
                if label != 'O' and label in self.target_entity_types and confidence >= self.confidence_threshold:
                    if current_entity is None or current_entity['entity_type'] != label:
                        if current_entity is not None and entity_confidences:
                            current_entity['confidence'] = sum(entity_confidences) / len(entity_confidences)
                            
                        current_entity = {
                            'entity_type': label, 
                            'start_position': index, 
                            'end_position': index+1,
                            'confidence': confidence
                        }
                        entity_confidences = [confidence]
                        ml_entities.append(current_entity)
                    elif index == current_entity['end_position']:
                        current_entity['end_position'] = index+1
                        entity_confidences.append(confidence)
                    else:
                        if current_entity is not None and entity_confidences:
                            current_entity['confidence'] = sum(entity_confidences) / len(entity_confidences)
                            
                        current_entity = {
                            'entity_type': label, 
                            'start_position': index, 
                            'end_position': index+1,
                            'confidence': confidence
                        }
                        entity_confidences = [confidence]
                        ml_entities.append(current_entity)
                else:
                    if current_entity is not None and entity_confidences:
                        current_entity['confidence'] = sum(entity_confidences) / len(entity_confidences)
                    current_entity = None
                    entity_confidences = []
        
        if current_entity is not None and entity_confidences:
            current_entity['confidence'] = sum(entity_confidences) / len(entity_confidences)
            
        combined_entities = rule_based_entities + ml_entities
        
        filtered_entities = []
        for entity in combined_entities:
            entity_type = entity['entity_type']
            start = entity['start_position']
            end = entity['end_position']
            entity_text = text[start:end].strip()
            
            min_length = self.min_entity_length.get(entity_type, 2)
            if len(entity_text) >= min_length:
                filtered_entities.append(entity)
        
        filtered_entities.sort(key=lambda x: x['start_position'])
        final_entities = []
        
        for entity in filtered_entities:
            overlap_idx = -1
            for i, existing in enumerate(final_entities):
                if (entity['start_position'] < existing['end_position'] and 
                    entity['end_position'] > existing['start_position']):
                    overlap_idx = i
                    if entity.get('confidence', 0) > existing.get('confidence', 0):
                        final_entities[i] = entity
                    break
            
            if overlap_idx == -1:  
                final_entities.append(entity)
        
        return final_entities

    def anonymize_text(self, text, location_list=None, company_list=None):
       
        if location_list:
            self.gazetteers['LOCATION'].update([loc.strip() for loc in location_list if loc.strip()])
        if company_list:
            self.gazetteers['ORGANIZATION'].update([comp.strip() for comp in company_list if comp.strip()])
        
        entities = self.predict(text)
        
        entities.sort(key=lambda x: (-x.get('confidence', 0), -x['start_position']))
        
        entity_counters = {etype.upper(): 1 for etype in self.target_entity_types}
        anonymized_text = text
        replacement_map = {}
        
        for entity in entities:
            entity_type = entity['entity_type'].upper()
            start = entity['start_position']
            end = entity['end_position']
            original_text = text[start:end]
            confidence = entity.get('confidence', 0)
            
            min_length = self.min_entity_length.get(entity_type, 2)
            if len(original_text.strip()) < min_length:
                continue
                
            if confidence < self.confidence_threshold:
                continue
                
            placeholder = f"<{entity_type}_{entity_counters[entity_type]}>"
            entity_counters[entity_type] += 1
            
            anonymized_text = anonymized_text[:start] + placeholder + anonymized_text[end:]
            replacement_map[placeholder] = original_text
            
        return anonymized_text, replacement_map

    def evaluate(self, X_test, y_test):
        
        y_pred = self.model.predict(X_test)
        self.report_focused_metrics(y_test, y_pred)
        
        return classification_report(y_test, y_pred, output_dict=True)

    def save_model(self, model_path='custom_ner_model.pkl'):
        
        with open(model_path, 'wb') as f:
            pickle.dump({
                'model': self.model, 
                'entity_types': self.entity_types,
                'target_entity_types': self.target_entity_types,
                'gazetteers': self.gazetteers,
                'patterns': self.patterns,
                'min_entity_length': self.min_entity_length,
                'confidence_threshold': self.confidence_threshold
            }, f)
            
        print(f"Model saved to {model_path}")

    def load_model(self, model_path='custom_ner_model.pkl'):
        try:
            with open(model_path, 'rb') as f:
                data = pickle.load(f)
                self.model = data['model']
                self.entity_types = data['entity_types']
                if 'target_entity_types' in data:
                    self.target_entity_types = data['target_entity_types']
                if 'gazetteers' in data:
                    self.gazetteers = data['gazetteers']
                if 'patterns' in data:
                    self.patterns = data['patterns']
                if 'min_entity_length' in data:
                    self.min_entity_length = data['min_entity_length']
                if 'confidence_threshold' in data:
                    self.confidence_threshold = data['confidence_threshold']
                    
            print(f"Model loaded from {model_path}")
            return True
        except Exception as e:
            print(f"Error loading model: {e}")
            return False

In [57]:
import os
import sys

def main():
    ner_model = Improved_CustomNERModel()
    
    try:
        locations = ner_model.load_entity_list("locations.txt")
    except Exception as e:
        locations = []
        
    try:
        companies = ner_model.load_entity_list("companies.txt")
    except Exception as e:
        companies = []
    
    train_file = "dataset_new1.txt"
    test_file = "test_2.txt"
    model_file = "focused_ner_model.pkl"
    
    print(f"Using training data from: {train_file}")
    print(f"Using test data from: {test_file}")
    
    if os.path.exists(model_file):
        ner_model.load_model(model_file)
    else:
        print("No existing model found. Training a new one.")
        
        train_data = ner_model.load_data(train_file)
        print(f"Loaded {len(train_data)} training examples.")
        
        tokens, labels = ner_model.preprocess_data(train_data)
        X, y = ner_model.prepare_training_data(tokens, labels)
        
        ner_model.train(X, y)
        
        ner_model.save_model(model_file)
    
    test_data = ner_model.load_data(test_file, is_json=False)
    
    if test_data:
        sample_text = test_data[0].get('full_text', '')
        if sample_text:
            # Anonymize text
            anonymized_text, replacements = ner_model.anonymize_text(sample_text, location_list=locations, company_list=companies)
            
            print("\nSample text:")
            print(sample_text)
            print("\nAnonymized text:")
            print(anonymized_text)
            print("\nReplacements:")
            for placeholder, original in replacements.items():
                print(f"{placeholder}: {original}")
    else:
        print("No test data available.")

if __name__ == "__main__":
    main()

Loaded 210 entities from locations.txt
Loaded 435 entities from companies.txt
Using training data from: dataset_new1.txt
Using test data from: test_2.txt
Model loaded from focused_ner_model.pkl
Loaded 61 text paragraphs from test_2.txt

Sample text:
Apple was founded by Steve Jobs
Unilever (ULVR LN) 
Equities 
Personal Products 
Email: <saurabh.mishra2@gds.com>
Head of Consumer Staples Research, Europe 
HSBC Bank plc jeremy.fialko@hsbc.com +44 20 7991 1562 
Sameer Lam* 
Analyst, Consumer Staples HSBC Bank plc sameer.lam@hsbc.com 
7238394647

Anonymized text:
<ORGANIZATION_2> was founded by Steve Jobs
<ORGANIZATION_1> (ULVR LN) 
Equities 
Personal Products 
Email: <<EMAIL_3>>
Head of Consumer Staples Research, Europe 
HSBC B<LOCATION_1>c <EMAIL_2> +44 20 7991 1562 
Sameer Lam* 
Analyst, Consumer Staples HSBC Bank plc <EMAIL_1> 
<PHONE_NUMBER_1>

Replacements:
<PHONE_NUMBER_1>: 7238394647
<EMAIL_1>: sameer.lam@hsbc.com
<EMAIL_2>: jeremy.fialko@hsbc.com
<EMAIL_3>: saurabh.mishra2@gds.com
